# Sel 1: Instalasi dan impor

In [1]:
%pip install pandas tqdm

import time
import urllib.request
from ftplib import FTP
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor

import pandas as pd
from tqdm.auto import tqdm

Note: you may need to restart the kernel to use updated packages.


/opt/anaconda3/envs/biochem/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Sel 2: Konfigurasi

In [2]:
DATA_DIR = Path("data_bvbrc")
FAA_DIR = DATA_DIR / "faa"
FAA_DIR.mkdir(parents=True, exist_ok=True)

SPECIES_KEYWORD = "Mycobacterium tuberculosis"   # ganti sesuai target
ANTIBIOTIC = "rifampicin"                        # ganti sesuai target (huruf kecil)
MAX_GENOMES = 20                                 # batasi dulu untuk uji coba, isi None untuk semua
N_WORKERS = 4                                    # jangan terlalu besar agar server FTP tidak terbebani

In [3]:
from ftplib import FTP, FTP_TLS

class ReusedSessionFTP_TLS(FTP_TLS):
    """FTP_TLS yang memakai ulang sesi TLS kanal kontrol pada kanal data."""
    def ntransfercmd(self, cmd, rest=None):
        conn, size = FTP.ntransfercmd(self, cmd, rest)
        if self._prot_p:
            conn = self.context.wrap_socket(
                conn, server_hostname=self.host, session=self.sock.session
            )
        return conn, size

def ftps_connect(host="ftp.bvbrc.org", timeout=60):
    ftp = ReusedSessionFTP_TLS()
    ftp.connect(host, 21, timeout=timeout)
    ftp.login()        # anonymous
    ftp.prot_p()       # enkripsi kanal data
    return ftp

def ftps_download(ftp, remote_path, local_path):
    local_path = Path(local_path)
    tmp = local_path.with_suffix(local_path.suffix + ".part")
    with open(tmp, "wb") as f:
        ftp.retrbinary(f"RETR {remote_path}", f.write)
    tmp.replace(local_path)

# Sel 3: Unduh tabel fenotipe AMR

In [4]:
AMR_REMOTE = "/RELEASE_NOTES/PATRIC_genomes_AMR.txt"
AMR_FILE = DATA_DIR / "PATRIC_genomes_AMR.txt"

if not AMR_FILE.exists():
    ftp = ftps_connect()
    try:
        ftps_download(ftp, AMR_REMOTE, AMR_FILE)
    finally:
        ftp.quit()

amr = pd.read_csv(AMR_FILE, sep="\t", dtype=str, low_memory=False)
amr.columns = amr.columns.str.strip().str.lower()
print(amr.shape)
print(list(amr.columns))
amr.head()

(1207585, 16)
['genome_id', 'genome_name', 'taxon_id', 'antibiotic', 'resistant_phenotype', 'measurement', 'measurement_sign', 'measurement_value', 'measurement_unit', 'laboratory_typing_method', 'laboratory_typing_method_version', 'laboratory_typing_platform', 'vendor', 'testing_standard', 'testing_standard_year', 'source']


,genome_id,genome_name,taxon_id,antibiotic,resistant_phenotype,measurement,measurement_sign,measurement_value,measurement_unit,laboratory_typing_method,laboratory_typing_method_version,laboratory_typing_platform,vendor,testing_standard,testing_standard_year,source
0,562.144637,"""Escherichia coli """,562,amoxicillin/clavulanic acid,NaN,2,NaN,2,mg/L,MIC,NaN,NaN,NaN,CLSI,2018,NaN
1,562.144637,"""Escherichia coli """,562,ampicillin,NaN,<=1,<=,1,mg/L,MIC,NaN,NaN,NaN,CLSI,2018,NaN
2,562.144637,"""Escherichia coli """,562,azithromycin,NaN,2,NaN,2,mg/L,MIC,NaN,NaN,NaN,CLSI,2018,NaN
3,562.144637,"""Escherichia coli """,562,cefoxitin,NaN,2,NaN,2,mg/L,MIC,NaN,NaN,NaN,CLSI,2018,NaN
4,562.144637,"""Escherichia coli """,562,ceftriaxone,NaN,<=0.25,<=,0.25,mg/L,MIC,NaN,NaN,NaN,CLSI,2018,NaN


# Sel 4: Ringkasan jumlah data per spesies dan antibiotik

In [5]:
need = {"genome_id", "genome_name", "antibiotic", "resistant_phenotype"}
missing = need - set(amr.columns)
assert not missing, f"Kolom tidak ditemukan: {missing}. Cek nama kolom pada output Sel 3."

sir = amr[amr["resistant_phenotype"].isin(["Susceptible", "Resistant"])]
summary = (sir.groupby(["genome_name", "antibiotic", "resistant_phenotype"])["genome_id"]
              .nunique().unstack(fill_value=0))
summary["total"] = summary.sum(axis=1)
summary.sort_values("total", ascending=False).head(50)

resistant_phenotype                                                   Resistant  \
genome_name                            antibiotic                                 
Streptococcus pneumoniae strain GPSC17 penicillin                            97   
Streptococcus pneumoniae strain GPSC3  trimethoprim/sulfamethoxazole          7   
                                       erythromycin                           8   
Streptococcus pneumoniae strain GPSC5  penicillin                            88   
Streptococcus pneumoniae strain GPSC10 penicillin                            89   
Streptococcus pneumoniae strain GPSC14 erythromycin                          89   
Streptococcus pneumoniae strain GPSC10 tetracycline                          84   
Streptococcus pneumoniae strain GPSC14 tetracycline                          86   
Streptococcus pneumoniae strain GPSC2  trimethoprim/sulfamethoxazole         18   
Streptococcus pneumoniae strain GPSC10 erythromycin                          76   
                                       trimethoprim/sulfamethoxazole         78   
Streptococcus pneumoniae strain GPSC3  penicillin                             9   
Streptococcus pneumoniae strain GPSC17 trimethoprim/sulfamethoxazole         76   
Streptococcus pneumoniae strain GPSC18 penicillin                            61   
                                       erythromycin                          73   
Streptococcus pneumoniae strain GPSC14 penicillin                            65   
Streptococcus pneumoniae strain GPSC8  erythromycin                           0   
Streptococcus pneumoniae strain GPSC1  penicillin                            69   
Streptococcus pneumoniae strain GPSC5  erythromycin                          17   
Streptococcus pneumoniae strain GPSC2  erythromycin                           2   
Streptococcus pneumoniae strain GPSC14 trimethoprim/sulfamethoxazole         70   
Streptococcus pneumoniae strain GPSC5  trimethoprim/sulfamethoxazole         47   
Streptococcus pneumoniae strain GPSC55 erythromycin                           0   
                                       chloramphenicol                        0   
Streptococcus pneumoniae strain GPSC3  chloramphenicol                        1   
Streptococcus pneumoniae strain GPSC55 penicillin                            65   
Streptococcus pneumoniae strain GPSC3  tetracycline                           3   
Streptococcus pneumoniae strain GPSC17 erythromycin                           3   
Streptococcus pneumoniae strain GPSC1  erythromycin                          59   
Streptococcus pneumoniae strain GPSC55 trimethoprim/sulfamethoxazole          2   
Streptococcus pneumoniae strain GPSC8  chloramphenicol                        0   
Streptococcus pneumoniae strain GPSC18 tetracycline                          37   
Streptococcus pneumoniae strain GPSC1  trimethoprim/sulfamethoxazole         60   
Streptococcus pneumoniae strain GPSC8  trimethoprim/sulfamethoxazole         28   
Streptococcus pneumoniae strain GPSC18 trimethoprim/sulfamethoxazole         59   
Streptococcus pneumoniae strain GPSC41 penicillin                            48   
Streptococcus pneumoniae strain GPSC55 tetracycline                           4   
Streptococcus pneumoniae strain GPSC8  penicillin                             4   
Streptococcus pneumoniae strain GPSC17 chloramphenicol                        0   
Streptococcus pneumoniae strain GPSC8  tetracycline                           6   
Streptococcus pneumoniae strain GPSC5  chloramphenicol                        2   
Streptococcus pneumoniae strain GPSC13 trimethoprim/sulfamethoxazole         16   
Streptococcus pneumoniae strain GPSC5  tetracycline                           5   
Streptococcus pneumoniae strain GPSC41 trimethoprim/sulfamethoxazole         51   
Streptococcus pneumoniae strain GPSC2  chloramphenicol                        1   
Streptococcus pneumoniae strain GPSC1  tetracycline                          43   
Streptococcus pn

# Sel 5: Filter data sesuai target

In [6]:
# Isi dengan nama antibiotik persis seperti pada output sel diagnosis (huruf kecil)
ANTIBIOTIC_ALIASES = ["rifampin", "rifampicin"]

df = amr.copy()
print("Awal:", len(df))

# 1) Spesies
df = df[df["genome_name"].str.contains(SPECIES_KEYWORD, case=False, regex=False, na=False)]
print("Setelah filter spesies:", len(df))

# 2) Antibiotik (cocokkan salah satu alias)
pattern = "|".join(ANTIBIOTIC_ALIASES)
df = df[df["antibiotic"].str.lower().str.contains(pattern, regex=True, na=False)]
print("Setelah filter antibiotik:", len(df))

# 3) Hanya Susceptible dan Resistant
df["resistant_phenotype"] = df["resistant_phenotype"].str.strip().str.capitalize()
df = df[df["resistant_phenotype"].isin(["Susceptible", "Resistant"])]
print("Setelah filter S/R:", len(df))

# 4) Buang label hasil prediksi komputasi (jika kolomnya ada)
if "computational_method" in df.columns:
    before = len(df)
    df = df[df["computational_method"].isna() | (df["computational_method"].str.strip() == "")]
    print(f"Setelah buang prediksi komputasi: {len(df)} (dibuang {before - len(df)})")

assert len(df) > 0, "Hasil filter kosong. Lihat tahap mana yang turun ke 0 di atas."

# 5) Buang genom dengan label bertentangan, lalu hapus duplikat
n_label = df.groupby("genome_id")["resistant_phenotype"].nunique()
conflict = n_label[n_label > 1].index
print("Genom dengan label bertentangan (dibuang):", len(conflict))
df = df[~df["genome_id"].isin(conflict)]

labels = (df[["genome_id", "genome_name", "antibiotic", "resistant_phenotype"]]
          .drop_duplicates("genome_id").reset_index(drop=True))
labels["label"] = (labels["resistant_phenotype"] == "Resistant").astype(int)

print("\nDistribusi kelas (seluruh data terfilter):")
print(labels["resistant_phenotype"].value_counts())

if MAX_GENOMES:
    labels = labels.sample(n=min(MAX_GENOMES, len(labels)), random_state=42).reset_index(drop=True)
labels.to_csv(DATA_DIR / "labels.csv", index=False)
labels.head()

Awal: 1207585
Setelah filter spesies: 251897
Setelah filter antibiotik: 33880
Setelah filter S/R: 24282
Genom dengan label bertentangan (dibuang): 1

Distribusi kelas (seluruh data terfilter):
resistant_phenotype
Susceptible    16114
Resistant       7064
Name: count, dtype: int64


,genome_id,genome_name,antibiotic,resistant_phenotype,label
0,1773.1568,Mycobacterium tuberculosis strain 0268S,rifampin,Susceptible,0
1,1733.1247,Mycobacterium tuberculosis 0516M,rifampin,Susceptible,0
2,1733.11166,Mycobacterium tuberculosis 13s405,rifampin,Susceptible,0
3,1773.4741,Mycobacterium tuberculosis C00014898,rifampin,Susceptible,0
4,1733.4794,Mycobacterium tuberculosis 13.0612462,rifampin,Susceptible,0


# Sel 6: Verifikasi lokasi file protein di FTP

In [7]:
gid = labels["genome_id"].iloc[0]
ftp = ftps_connect()
try:
    ftp.cwd(f"/genomes/{gid}")
    print(ftp.nlst())
finally:
    ftp.quit()

['1773.1568.PATRIC.spgene.tab', '1773.1568.PATRIC.pathway.tab', '1773.1568.PATRIC.frn', '1773.1568.PATRIC.gff', '1773.1568.PATRIC.features.tab', '1773.1568.PATRIC.faa', '1773.1568.PATRIC.ffn', '1773.1568.fna', '1773.1568.PATRIC.subsystem.tab']


# Sel 7: Unduh file protein (.faa)

In [8]:
FAA_REMOTE_TEMPLATE = "/genomes/{gid}/{gid}.PATRIC.faa"

def worker(gids):
    ftp, ok_list, fail_list = None, [], []
    for gid in gids:
        out = FAA_DIR / f"{gid}.faa"
        if out.exists() and out.stat().st_size > 0:
            ok_list.append(gid)
            continue
        success = False
        for _ in range(3):
            try:
                if ftp is None:
                    ftp = ftps_connect()
                ftps_download(ftp, FAA_REMOTE_TEMPLATE.format(gid=gid), out)
                success = True
                break
            except Exception:
                try:
                    if ftp is not None:
                        ftp.close()
                except Exception:
                    pass
                ftp = None
                time.sleep(2)
        (ok_list if success else fail_list).append(gid)
    try:
        if ftp is not None:
            ftp.quit()
    except Exception:
        pass
    return ok_list, fail_list

gids = labels["genome_id"].tolist()
chunks = [gids[i::N_WORKERS] for i in range(N_WORKERS)]

with ThreadPoolExecutor(max_workers=N_WORKERS) as ex:
    results = list(ex.map(worker, chunks))

ok = [g for r in results for g in r[0]]
failed = [g for r in results for g in r[1]]
print("Berhasil:", len(ok), "| Gagal:", len(failed))
pd.Series(failed).to_csv(DATA_DIR / "failed_genomes.txt", index=False, header=False)

Berhasil: 20 | Gagal: 0


# Sel 8: Baca FASTA dan gabungkan dengan label

In [9]:
def read_fasta(path):
    header, seq = None, []
    with open(path) as f:
        for line in f:
            line = line.rstrip()
            if line.startswith(">"):
                if header is not None:
                    yield header, "".join(seq)
                header, seq = line[1:], []
            else:
                seq.append(line)
    if header is not None:
        yield header, "".join(seq)

rows = []
for gid in labels["genome_id"]:
    path = FAA_DIR / f"{gid}.faa"
    if not path.exists():
        continue
    for header, seq in read_fasta(path):
        rows.append((gid, header, seq))

proteins = pd.DataFrame(rows, columns=["genome_id", "header", "sequence"])
data = proteins.merge(labels[["genome_id", "resistant_phenotype", "label"]],
                      on="genome_id", how="inner")

print(data.shape)
print(data["genome_id"].nunique(), "genom")
data.to_csv(DATA_DIR / "proteins_labeled.csv.gz", index=False, compression="gzip")
data.head()

(88952, 5)
20 genom


,genome_id,header,sequence,resistant_phenotype,label
0,1773.1568,fig|1773.1568.peg.1 PE_PGRS family protein =...,MSFLIASPEALAATATYLTGIGSAISAANAVAAAPTTEILAAGTDE...,Susceptible,0
1,1773.1568,fig|1773.1568.peg.2 Acid and phagosome regul...,MAKVTAARPEPSAAWAEARRRVRQRREDMLRHPAFLSKQLPAEPAD...,Susceptible,0
2,1773.1568,fig|1773.1568.peg.3|ERS094118_00003 Oligopep...,MSGATVGAREITIRGVVLGALITLVFTAANVYLGLRVGLTFATSIP...,Susceptible,0
3,1773.1568,fig|1773.1568.peg.4|ERS094118_00004 Gamma-gl...,MLSLSGCGGFHAGAPSTAGPCEIVPNGTPAPKTPPATVPSSRNLAT...,Susceptible,0
4,1773.1568,fig|1773.1568.peg.5|ERS094118_00005 Sirohydr...,MTAPATMQSAAMLRSGAIEAPPATMQSAAMRWGHLPLAEESGTIAP...,Susceptible,0
